In [ ]:
from scipy.stats import ttest_ind
import pandas as pd
import numpy as np

# Load the dataset
try:
    df_cleaned = pd.read_csv("cadcs_live_project_clean_data.csv")
    print("Dataset 'cadcs_live_project_clean_data.csv' loaded successfully.")
except FileNotFoundError:
    print("Error: 'cadcs_live_project_clean_data.csv' not found. Please ensure the file is in the correct directory.")
    exit() # Exit if the file is not found

# --- Data Cleaning and Feature Engineering (Placeholder for your previous steps) ---
# Ensure 'is_peak_hour' is created if not already
if 'is_peak_hour' not in df_cleaned.columns:
    print("Warning: 'is_peak_hour' not found. Attempting to create 'is_peak_hour' from 'start_time'.")
    if 'start_time' in df_cleaned.columns:
        df_cleaned['start_time'] = pd.to_datetime(df_cleaned['start_time'], errors='coerce')
        df_cleaned['hour'] = df_cleaned['start_time'].dt.hour
        # Define peak hours (e.g., 5 PM to 9 PM) - adjust this as per your definition
        df_cleaned['is_peak_hour'] = ((df_cleaned['hour'] >= 17) & (df_cleaned['hour'] <= 21)).astype(int)
        df_cleaned.drop(columns=['hour'], inplace=True) # Clean up temporary hour column
    else:
        print("Could not create 'is_peak_hour' as 'start_time' column is missing. 'is_peak_hour' comparison will be skipped.")
        # Set to None to indicate it's not available for the test
        df_cleaned['is_peak_hour'] = None


# Ensure 'charging_duration_min' is numeric
if 'charging_duration_min' in df_cleaned.columns:
    df_cleaned['charging_duration_min'] = pd.to_numeric(df_cleaned['charging_duration_min'], errors='coerce')
    df_cleaned.dropna(subset=['charging_duration_min'], inplace=True)
else:
    print("Error: 'charging_duration_min' column not found in df_cleaned. Exiting.")
    exit()

# Ensure 'charger_type' is present for the second comparison
if 'charger_type' not in df_cleaned.columns:
    print("Error: 'charger_type' column not found in df_cleaned. Charger type comparison will be skipped.")


print("\n--- Applying Independent Samples t-test to 'charging_duration_min' ---")

# Example 1: Comparing 'charging_duration_min' based on 'is_peak_hour'
if 'is_peak_hour' in df_cleaned.columns and df_cleaned['is_peak_hour'] is not None and 'charging_duration_min' in df_cleaned.columns:
    group_peak = df_cleaned[df_cleaned['is_peak_hour'] == 1]['charging_duration_min']
    group_non_peak = df_cleaned[df_cleaned['is_peak_hour'] == 0]['charging_duration_min']

    if not group_peak.empty and not group_non_peak.empty:
        # Perform t-test. Use equal_var=False (Welch's t-test) as a safer default if variances are unknown or unequal.
        stat, p_value = ttest_ind(group_peak, group_non_peak, equal_var=False)
        print(f"\nComparison: Charging Duration (Peak vs Non-Peak Hours)")
        print(f"t-statistic: {stat:.4f}, p-value: {p_value:.4f}")

        if p_value < 0.05: # Using a significance level (alpha) of 0.05
            print("Conclusion: Reject H0. There is a statistically significant difference in average charging duration between peak and non-peak hours.")
        else:
            print("Conclusion: Fail to reject H0. No statistically significant difference in average charging duration between peak and non-peak hours.")
    else:
        print("Groups for 'is_peak_hour' comparison are empty. Check data for this column and its values (0 and 1).")
else:
    print("'is_peak_hour' or 'charging_duration_min' column not found or 'is_peak_hour' not properly generated in df_cleaned for this comparison.")


# Example 2: Comparing 'charging_duration_min' between two specific 'charger_type' categories
# Using the values you provided: 'AC FAST', 'AC SLOW', 'DC FAST'
if 'charger_type' in df_cleaned.columns and 'charging_duration_min' in df_cleaned.columns:
    # Set the two charger types you want to compare
    charger_type1_name = 'DC_FAST'
    charger_type2_name = 'AC_FAST'

    # Select charging durations for the first charger type
    group_charger_type1 = df_cleaned[df_cleaned['charger_type'] == charger_type1_name]['charging_duration_min']
    # Select charging durations for the second charger type
    group_charger_type2 = df_cleaned[df_cleaned['charger_type'] == charger_type2_name]['charging_duration_min']

    if not group_charger_type1.empty and not group_charger_type2.empty:
        stat, p_value = ttest_ind(group_charger_type1, group_charger_type2, equal_var=False)
        print(f"\nComparison: Charging Duration ({charger_type1_name} vs {charger_type2_name} Chargers)")
        print(f"t-statistic: {stat:.4f}, p-value: {p_value:.4f}")

        if p_value < 0.05:
            print(f"Conclusion: Reject H0. There is a statistically significant difference in average charging duration between {charger_type1_name} and {charger_type2_name} chargers.")
        else:
            print(f"Conclusion: Fail to reject H0. No statistically significant difference in average charging duration between {charger_type1_name} and {charger_type2_name} chargers.")
    else:
        print(f"One or both specific charger type groups ('{charger_type1_name}' or '{charger_type2_name}') are empty. Check data for 'charger_type' column and the specified names.")
else:
    print("'charger_type' or 'charging_duration_min' column not found in df_cleaned for this comparison.")

Dataset 'cadcs_live_project_clean_data.csv' loaded successfully.

--- Applying Independent Samples t-test to 'charging_duration_min' ---

Comparison: Charging Duration (Peak vs Non-Peak Hours)
t-statistic: -1.6570, p-value: 0.0975
Conclusion: Fail to reject H0. No statistically significant difference in average charging duration between peak and non-peak hours.

Comparison: Charging Duration (DC_FAST vs AC_FAST Chargers)
t-statistic: -80.6226, p-value: 0.0000
Conclusion: Reject H0. There is a statistically significant difference in average charging duration between DC_FAST and AC_FAST chargers.


In [ ]:
if 'charger_type' in df_cleaned.columns and 'charging_duration_min' in df_cleaned.columns:
    # Set the two charger types you want to compare
    charger_type1_name = 'AC_SLOW'
    charger_type2_name = 'AC_FAST'

    # Select charging durations for the first charger type
    group_charger_type1 = df_cleaned[df_cleaned['charger_type'] == charger_type1_name]['charging_duration_min']
    # Select charging durations for the second charger type
    group_charger_type2 = df_cleaned[df_cleaned['charger_type'] == charger_type2_name]['charging_duration_min']

    if not group_charger_type1.empty and not group_charger_type2.empty:
        stat, p_value = ttest_ind(group_charger_type1, group_charger_type2, equal_var=False)
        print(f"\nComparison: Charging Duration ({charger_type1_name} vs {charger_type2_name} Chargers)")
        print(f"t-statistic: {stat:.4f}, p-value: {p_value:.4f}")

        if p_value < 0.05:
            print(f"Conclusion: Reject H0. There is a statistically significant difference in average charging duration between {charger_type1_name} and {charger_type2_name} chargers.")
        else:
            print(f"Conclusion: Fail to reject H0. No statistically significant difference in average charging duration between {charger_type1_name} and {charger_type2_name} chargers.")
    else:
        print(f"One or both specific charger type groups ('{charger_type1_name}' or '{charger_type2_name}') are empty. Check data for 'charger_type' column and the specified names.")
else:
    print("'charger_type' or 'charging_duration_min' column not found in df_cleaned for this comparison.")


Comparison: Charging Duration (AC_SLOW vs AC_FAST Chargers)
t-statistic: 110.2943, p-value: 0.0000
Conclusion: Reject H0. There is a statistically significant difference in average charging duration between AC_SLOW and AC_FAST chargers.


In [ ]:
from scipy.stats import f_oneway
import pandas as pd
import numpy as np

# Load the dataset
try:
    df_cleaned = pd.read_csv("cadcs_live_project_clean_data.csv")
    print("Dataset 'cadcs_live_project_clean_data.csv' loaded successfully.")
except FileNotFoundError:
    print("Error: 'cadcs_live_project_clean_data.csv' not found. Please ensure the file is in the correct directory.")
    exit() # Exit if the file is not found

# --- Data Cleaning and Preprocessing for ANOVA ---
# Ensure 'charging_duration_min' is numeric and handle missing values
if 'charging_duration_min' in df_cleaned.columns:
    df_cleaned['charging_duration_min'] = pd.to_numeric(df_cleaned['charging_duration_min'], errors='coerce')
    # Drop rows where 'charging_duration_min' is NaN, as ANOVA requires complete cases
    df_cleaned.dropna(subset=['charging_duration_min'], inplace=True)
else:
    print("Error: 'charging_duration_min' column not found in df_cleaned. Exiting.")
    exit()

# It's good practice to also handle any potential NaNs in the grouping column ('vehicle_make')
# if you expect them, as f_oneway will error on NaN values.
# For simplicity, we'll drop rows with NaN in 'vehicle_make' for the ANOVA test.
if 'vehicle_make' in df_cleaned.columns:
    df_cleaned.dropna(subset=['vehicle_make'], inplace=True)
else:
    print("Warning: 'vehicle_make' column not found in df_cleaned. ANOVA for vehicle make will be skipped.")


print("\n--- Applying One-Way ANOVA to 'charging_duration_min' ---")

# Example: Comparing 'charging_duration_min' across different 'vehicle_make'
if 'vehicle_make' in df_cleaned.columns and 'charging_duration_min' in df_cleaned.columns:
    unique_makes = df_cleaned['vehicle_make'].unique()

    # Create a list to hold the data groups for ANOVA
    groups = []

    # Iterate through unique vehicle makes and collect their charging durations
    for make in unique_makes:
        # Get charging durations for the current make
        make_data = df_cleaned[df_cleaned['vehicle_make'] == make]['charging_duration_min']

        # Only include groups with more than one observation (required for variance calculation)
        if len(make_data) > 1:
            groups.append(make_data)
        else:
            print(f"Skipping vehicle_make '{make}' due to insufficient data (less than 2 samples).")

    # We need at least 3 groups for ANOVA
    if len(groups) >= 3:
        stat, p_value = f_oneway(*groups) # The asterisk unpacks the list of Series/arrays
        print(f"\nComparison: Charging Duration across different Vehicle Makes")
        print(f"F-statistic: {stat:.4f}, p-value: {p_value:.4f}")

        if p_value < 0.05:
            print("Conclusion: Reject H0. There is a statistically significant difference in average charging duration across at least some vehicle makes.")
            # If significant, you might follow up with post-hoc tests (e.g., Tukey HSD)
            # from statsmodels.stats.multicomp import pairwise_tukeyhsd
            # tukey_result = pairwise_tukeyhsd(endog=df_cleaned['charging_duration_min'],
            #                                   groups=df_cleaned['vehicle_make'],
            #                                   alpha=0.05)
            # print("\nTukey HSD Post-hoc Test:")
            # print(tukey_result)
        else:
            print("Conclusion: Fail to reject H0. No statistically significant difference in average charging duration across vehicle makes.")
    else:
        print(f"Not enough unique 'vehicle_make' categories with sufficient data (need at least 3 groups with >1 sample each) to perform ANOVA. Found {len(groups)} valid groups.")
else:
    print("'vehicle_make' or 'charging_duration_min' column not found or not suitable in df_cleaned for ANOVA.")

Dataset 'cadcs_live_project_clean_data.csv' loaded successfully.

--- Applying One-Way ANOVA to 'charging_duration_min' ---

Comparison: Charging Duration across different Vehicle Makes
F-statistic: 3844.9845, p-value: 0.0000
Conclusion: Reject H0. There is a statistically significant difference in average charging duration across at least some vehicle makes.


In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

# Load your cleaned dataset
# Ensure 'cadcs_live_project_clean_data.csv' contains the original categorical columns
df_cleaned = pd.read_csv('/content/Dataset_BEV_CADCS_Project.csv')

print("\n--- Applying Chi-Squared Test of Independence (with Original Categorical Columns) ---")

# Example 1: Check association between 'charger_type' and 'payment_method'
# This assumes 'charger_type' and 'payment_method' are present as original categorical columns.
if 'charger_type' in df_cleaned.columns and 'payment_method' in df_cleaned.columns:
    # Create a contingency table (cross-tabulation)
    # This counts the occurrences of each combination of categories
    contingency_table_charger_payment = pd.crosstab(df_cleaned['charger_type'], df_cleaned['payment_method'])

    print(f"\nContingency Table (charger_type vs payment_method):\n{contingency_table_charger_payment}")

    # Check if the contingency table is valid for the test
    if contingency_table_charger_payment.empty or contingency_table_charger_payment.sum().sum() == 0:
        print("Contingency table is empty or has no data. Cannot perform Chi-Squared test.")
    else:
        # Perform the Chi-Squared test
        stat, p_value, dof, expected = chi2_contingency(contingency_table_charger_payment)

        print(f"\nChi-Squared Test (charger_type vs payment_method):")
        print(f"Chi2 Statistic: {stat:.4f}, p-value: {p_value:.4f}")
        print(f"Degrees of Freedom (dof): {dof}")
        # print(f"Expected Frequencies Table:\n{expected}") # Uncomment to see expected values

        if p_value < 0.05: # Using a common significance level (alpha) of 0.05
            print("Conclusion: Reject H0. There is a statistically significant association between charger type and payment method.")
        else:
            print("Conclusion: Fail to reject H0. No statistically significant association between charger type and payment method.")
else:
    print("'charger_type' or 'payment_method' column not found in the loaded df_cleaned. Please check column names.")


# --- You can repeat this for other pairs of original categorical columns ---

# Example 2: Check association between 'vehicle_make' and 'charger_type'
if 'vehicle_make' in df_cleaned.columns and 'charger_type' in df_cleaned.columns:
    contingency_table_vehicle_charger = pd.crosstab(df_cleaned['vehicle_make'], df_cleaned['charger_type'])
    print(f"\nContingency Table (vehicle_make vs charger_type):\n{contingency_table_vehicle_charger}")

    if contingency_table_vehicle_charger.empty or contingency_table_vehicle_charger.sum().sum() == 0:
        print("Contingency table is empty or has no data. Cannot perform Chi-Squared test.")
    else:
        stat, p_value, dof, expected = chi2_contingency(contingency_table_vehicle_charger)
        print(f"\nChi-Squared Test (vehicle_make vs charger_type):")
        print(f"Chi2 Statistic: {stat:.4f}, p-value: {p_value:.4f}")
        if p_value < 0.05:
            print("Conclusion: Reject H0. There is a statistically significant association between vehicle make and charger type.")
        else:
            print("Conclusion: Fail to reject H0. No statistically significant association between vehicle make and charger type.")
else:
    print("'vehicle_make' or 'charger_type' column not found in the loaded df_cleaned.")


# Example 3: Check association between 'is_weekend' and 'weather_condition' (if 'weather_condition' is original categorical)
# Assuming 'is_weekend' is 0/1, it's already categorical-like.
if 'is_weekend' in df_cleaned.columns and 'weather_condition' in df_cleaned.columns:
    contingency_table_weekend_weather = pd.crosstab(df_cleaned['is_weekend'], df_cleaned['weather_condition'])
    print(f"\nContingency Table (is_weekend vs weather_condition):\n{contingency_table_weekend_weather}")

    if contingency_table_weekend_weather.empty or contingency_table_weekend_weather.sum().sum() == 0:
        print("Contingency table is empty or has no data. Cannot perform Chi-Squared test.")
    else:
        stat, p_value, dof, expected = chi2_contingency(contingency_table_weekend_weather)
        print(f"\nChi-Squared Test (is_weekend vs weather_condition):")
        print(f"Chi2 Statistic: {stat:.4f}, p-value: {p_value:.4f}")
        if p_value < 0.05:
            print("Conclusion: Reject H0. There is a statistically significant association between weekend and weather condition.")
        else:
            print("Conclusion: Fail to reject H0. No statistically significant association between weekend and weather condition.")
else:
    print("'is_weekend' or 'weather_condition' column not found in the loaded df_cleaned.")


--- Applying Chi-Squared Test of Independence (with Original Categorical Columns) ---

Contingency Table (charger_type vs payment_method):
payment_method  Card  Cash  Mobile_App  RFID  Subscription   UPI  Wallet
charger_type                                                            
AC_Fast         2703  2643        2561  2687          2633  2645    2686
AC_Slow         6498  6425        6653  6483          6365  6389    6402
DC_Fast         1522  1553        1678  1592          1636  1585    1661

Chi-Squared Test (charger_type vs payment_method):
Chi2 Statistic: 20.6657, p-value: 0.0555
Degrees of Freedom (dof): 12
Conclusion: Fail to reject H0. No statistically significant association between charger type and payment method.

Contingency Table (vehicle_make vs charger_type):
charger_type  AC_Fast  AC_Slow  DC_Fast
vehicle_make                           
Audi               78      220       65
BMW                97      225       51
BYD               189      451      105
Bajaj    